# NB02 – Data Transformation

## Purpose

In NB01 I collected the raw product data and saved one JSON file per country in
the `data/raw/` folder. Those files are the raw API responses, which are nested
and not easy to analyse directly. Each file holds a list of products under a key
called `hits`, and inside every product the nutritional values sit in their own
dictionary called `nutriments`.

The goal of this notebook is to turn that raw data into a clean, flat table that
I can actually work with. For each country I go through every product in the file
and pull out the fields I need:

- the **barcode** (`code`)
- the **product name** (`product_name`)
- the **brand** (`brands`)
- the **sugar content** in grams per 100g (`sugars_100g`, which sits inside `nutriments`)
- the **country** the product was collected for

The country is the one column that does not come from the product record itself.
Each API request already filtered to a single country, so the response never
repeats that information product by product. I take it from the name of the file
each set of products was saved in, and stamp it onto every row.

The other thing this notebook has to handle is missing data. Many products in
Open Food Facts have been scanned but never filled in, so some have no name, no
brand, or no sugar value at all. I do not remove these rows here. They are kept
so that the amount of missing data can be counted per country in NB03, since how
much is missing is itself part of the answer.

Once everything is pulled out, I save the result as a **CSV file** in the
`data/processed/` folder. That CSV becomes the input for NB03, where I do the
actual analysis (comparing sugar content between countries with and without a
sugar tax).

I do not do any analysis here. This notebook only reshapes the data from nested
JSON into a clean table.

In [9]:
import json 
import pandas as pd 
from pathlib import Path

RAW_DIR = Path("../data/raw")

rows = []
for path in RAW_DIR.glob("*.json"): 
    country = path.stem # .stem basically just takes the name of the file without the type so it stores "italy"
    with open(RAW_DIR / f"{country}.json",  encoding='utf-8') as f: 
        sample = json.load(f)

        for reading in sample["hits"]: 
            nutriments = reading.get("nutriments", {})
            brands = reading.get("brands", [])

            rows.append({
                "country": country,
                "barcode": reading.get("code"),
                "product_name": reading.get("product_name"),
                "brand": brands[0] if brands else None,
                "sugar_content_per_100g:": nutriments.get("sugars_100g")
            })

df = pd.DataFrame(rows)

df.to_csv("../data/processed/sodas_clean.csv", index=False, encoding='utf-8')
df.head()

,country,barcode,product_name,brand,sugar_content_per_100g:
0,italy,0000503210227,None,None,NaN
1,italy,8002516010223,La classica,Tomarchio,11.0
2,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4
3,italy,4060800129680,Pepsi lemon,Pepsi,10.7
4,italy,4060800001771,Pepsi-cola,pepsi,10.9
